# XiDepth v2.0 — Track 1: ResNet-18 Baseline Training

This notebook trains the verified **ResNet-18 baseline** (14.7M parameters) on the KITTI Eigen-Zhou split using a free Kaggle NVIDIA Tesla T4 GPU (16 GB VRAM).

### Prerequisites:
1. In the right panel, select **Accelerator: GPU T4 x2** (or GPU T4 x1).
2. Click **+ Add Data** -> Search for `kitti-eigen-split` (by Grandmaster Awsaf) and attach it.

In [ ]:
# 1. Verify GPU availability
!nvidia-smi
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))


In [ ]:
# 2. Locate KITTI dataset on Kaggle
import os, glob
input_dirs = glob.glob("/kaggle/input/*kitti*")
print("Found candidate input dirs:", input_dirs)
dataset_dir = input_dirs[0] if input_dirs else "/kaggle/input/kitti-eigen-split"
print(f"Using dataset directory: {dataset_dir}")


In [ ]:
# 3. Clone or pull XiDepth-Monocular-2.0
!git clone https://github.com/ASMITSARKAR/XiDepth-Monocular-2.0.git xidepth_repo
%cd xidepth_repo
!pip install -r requirements.txt


In [ ]:
# 4. Run Track 1 Training (ResNet-18 baseline, 20 epochs, AMP FP16)
!python scripts/train.py \
    --model resnet18 \
    --pretrained \
    --dataset_dir "$dataset_dir" \
    --split_dir data/splits/eigen_zhou \
    --checkpoint_dir /kaggle/working/checkpoints_resnet18 \
    --batch_size 12 \
    --epochs 20 \
    --lr 1e-4 \
    --lr_step_size 15 \
    --num_workers 4 \
    --use_amp \
    --log_freq 100


In [ ]:
# 5. Evaluate trained ResNet-18 model on Eigen Benchmark
!python scripts/eval.py \
    --model resnet18 \
    --checkpoint /kaggle/working/checkpoints_resnet18/best_model.pth \
    --dataset_dir "$dataset_dir" \
    --split_file data/splits/eigen_zhou/test_files.txt \
    --gt_path data/gt_depths.npz \
    --output_json /kaggle/working/resnet18_results.json
